In [5]:
import json

with open("baseline_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

In [6]:
import json
from collections import Counter

In [7]:
def strict_validate(parsed):
    errors = []

    if not isinstance(parsed, dict):
        errors.append("not_a_dict")
        return False, errors

    required_keys = ["answer", "refused", "confidence"]

    for k in required_keys:
        if k not in parsed:
            errors.append(f"missing_{k}")

    if errors:
        return False, errors

    if not isinstance(parsed["answer"], str):
        errors.append("answer_not_string")

    if not isinstance(parsed["refused"], bool):
        errors.append("refused_not_boolean")

    if not isinstance(parsed["confidence"], (int, float)):
        errors.append("confidence_not_number")
    else:
        if not (0.0 <= float(parsed["confidence"]) <= 1.0):
            errors.append("confidence_out_of_range")

    return len(errors) == 0, errors

In [8]:
validated_results = []

for r in results:
    parsed = r["parsed"]

    if not r["json_block"]:
        validated_results.append({
            **r,
            "strict_valid": False,
            "strict_errors": ["no_json_block"]
        })
        continue

    try:
        parsed = json.loads(r["json_block"])
        ok, errors = strict_validate(parsed)
    except Exception:
        ok = False
        errors = ["json_parse_error"]

    validated_results.append({
        **r,
        "strict_valid": ok,
        "strict_errors": errors
    })

len(validated_results)

20

In [9]:
error_counter = Counter()

for r in validated_results:
    if not r["strict_valid"]:
        for e in r["strict_errors"]:
            error_counter[e] += 1

error_counter

Counter({'json_parse_error': 1,
         'answer_not_string': 1,
         'confidence_not_number': 1})

In [10]:
behavior_stats = {
    "hallucinations": 0,
    "wrong_refusals": 0,
    "correct_refusals": 0,
    "correct_answers": 0
}

for r in validated_results:
    if not r["strict_valid"]:
        continue

    parsed = json.loads(r["json_block"])

    if r["answerable"]:
        if parsed["refused"]:
            behavior_stats["wrong_refusals"] += 1
        else:
            behavior_stats["correct_answers"] += 1
    else:
        if parsed["refused"]:
            behavior_stats["correct_refusals"] += 1
        else:
            behavior_stats["hallucinations"] += 1

behavior_stats

{'hallucinations': 6,
 'wrong_refusals': 0,
 'correct_refusals': 2,
 'correct_answers': 10}